In [61]:
import pandas as pd 
import requests
from datetime import datetime
import mysql.connector
from sqlalchemy import create_engine
import urllib
import feedparser
import streamlit as st 
import plotly.express as px
import os
from dotenv import load_dotenv

### Flight Data (OpenSky API)

In [ ]:
load_dotenv()
USERNAME = os.getenv("OPENSKY_USERNAME")
PASSWORD = os.getenv("OPENSKY_PASSWORD")

URL = "https://opensky-network.org/api/states/all"

print(f"[{datetime.utcnow().strftime('%H:%M:%S')} UTC] Fetching live flight data from Opensky....")

response = requests.get(URL,auth = (USERNAME,PASSWORD))

if response.status_code == 200:
  data = response.json()
  states = data.get('states' , [])


  columns = [
        'icao24', 'callsign', 'origin_country', 'time_position', 
        'last_contact', 'longitude', 'latitude', 'baro_altitude', 
        'on_ground', 'velocity', 'true_track', 'vertical_rate', 
        'sensors', 'geo_altitude', 'squawk', 'spi', 'position_source'
    ]

  df_raw = pd.DataFrame(states , columns=columns)
  print(f"Raw data fetched Succesfully! TOTAL ROWS{len(df_raw)} and SHAPE IS {df_raw.shape}")

  display(df_raw.head(5))

else:
  print('Failed!!!')

C:\Users\HP\AppData\Local\Temp\ipykernel_2360\4179358103.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"[{datetime.utcnow().strftime('%H:%M:%S')} UTC] Fetching live flight data from Opensky....")


[03:50:53 UTC] Fetching live flight data from Opensky....
Raw data fetched Succesfully! TOTAL ROWS6443 and SHAPE IS (6443, 17)


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,80162c,AXB1672,India,1.793678e+09,1793677853,90.5848,26.1838,4953.00,False,182.38,105.88,-7.80,None,5204.46,0505,False,0
1,80162d,AXB2746,India,1.793678e+09,1793677854,87.0410,25.4311,8404.86,False,267.96,50.69,-8.45,None,8892.54,1314,False,0
2,801638,AXB2240,India,1.793678e+09,1793677854,77.3124,28.2552,3048.00,False,150.63,11.43,-0.65,None,3169.92,None,False,0
3,76eab9,UT7,Singapore,1.793678e+09,1793677631,103.9855,1.3727,NaN,True,9.26,202.50,NaN,None,NaN,None,False,0
4,801642,AXB1640,India,1.793678e+09,1793677831,76.4250,14.7646,10279.38,False,237.91,135.35,-4.88,None,10911.84,None,False,0


In [3]:
df_raw.shape

(6443, 17)

In [4]:
df_raw.columns

Index(['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact',
       'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity',
       'true_track', 'vertical_rate', 'sensors', 'geo_altitude', 'squawk',
       'spi', 'position_source'],
      dtype='object')

In [5]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6443 entries, 0 to 6442
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   icao24           6443 non-null   object 
 1   callsign         6443 non-null   object 
 2   origin_country   6443 non-null   object 
 3   time_position    6395 non-null   float64
 4   last_contact     6443 non-null   int64  
 5   longitude        6395 non-null   float64
 6   latitude         6395 non-null   float64
 7   baro_altitude    5622 non-null   float64
 8   on_ground        6443 non-null   bool   
 9   velocity         6442 non-null   float64
 10  true_track       6443 non-null   float64
 11  vertical_rate    5639 non-null   float64
 12  sensors          0 non-null      object 
 13  geo_altitude     5561 non-null   float64
 14  squawk           4053 non-null   object 
 15  spi              6443 non-null   bool   
 16  position_source  6443 non-null   int64  
dtypes: bool(2), fl

In [6]:
df_raw.describe()

,time_position,last_contact,longitude,latitude,baro_altitude,velocity,true_track,vertical_rate,geo_altitude,position_source
count,6.395000e+03,6.443000e+03,6395.000000,6395.000000,5622.000000,6442.000000,6443.000000,5639.000000,5561.000000,6443.0
mean,1.793678e+09,1.793678e+09,-21.274551,29.251456,7417.100896,166.446942,179.755321,-0.399617,7690.517544,0.0
std,1.909997e+03,5.949923e+01,97.797832,21.636769,4185.633147,96.820919,103.292245,4.890273,4321.969417,0.0
min,1.793526e+09,1.793678e+09,-170.629500,-46.853800,-121.920000,0.000000,0.000000,-23.080000,-30.480000,0.0
25%,1.793678e+09,1.793678e+09,-98.318300,26.557050,3232.785000,87.505000,90.000000,-0.650000,3360.420000,0.0
50%,1.793678e+09,1.793678e+09,-78.098900,35.111400,9239.250000,202.490000,180.140000,0.000000,9593.580000,0.0
75%,1.793678e+09,1.793678e+09,79.988900,40.791450,10972.800000,235.680000,270.980000,0.000000,11376.660000,0.0
max,1.793678e+09,1.793678e+09,179.691100,68.097900,14325.600000,2265.700000,359.880000,23.080000,14942.820000,0.0


### DATA CLEANING

In [7]:
df = df_raw.copy()

In [8]:
df.columns

Index(['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact',
       'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity',
       'true_track', 'vertical_rate', 'sensors', 'geo_altitude', 'squawk',
       'spi', 'position_source'],
      dtype='object')

In [9]:
df['callsign'] = df['callsign'].astype(str).str.strip()

In [10]:
active_flight = df[df['on_ground'] == False].copy()

In [11]:
active_flight.isnull().sum()

icao24                0
callsign              0
origin_country        0
time_position        48
last_contact          0
longitude            48
latitude             48
baro_altitude        33
on_ground             0
velocity              0
true_track            0
vertical_rate         6
sensors            5642
geo_altitude         81
squawk             1860
spi                   0
position_source       0
dtype: int64

In [12]:
relevant_cols = ['icao24', 'callsign', 'origin_country', 'longitude', 'latitude', 'velocity', 'geo_altitude', 'vertical_rate']
active_flight = active_flight[relevant_cols]

In [13]:
active_flight.isnull().sum()

icao24             0
callsign           0
origin_country     0
longitude         48
latitude          48
velocity           0
geo_altitude      81
vertical_rate      6
dtype: int64

In [14]:
active_flight = active_flight.dropna(subset=['callsign' , 'longitude' , 'latitude'])


In [15]:
active_flight.isnull().sum()

icao24             0
callsign           0
origin_country     0
longitude          0
latitude           0
velocity           0
geo_altitude      49
vertical_rate      6
dtype: int64

In [16]:
active_flight = active_flight[active_flight['callsign'] != '']

In [17]:
active_flight['geo_altitude'] = active_flight['geo_altitude'].fillna(0)
active_flight['vertical_rate'] = active_flight['vertical_rate'].fillna(0)

In [18]:
active_flight.isnull().sum()

icao24            0
callsign          0
origin_country    0
longitude         0
latitude          0
velocity          0
geo_altitude      0
vertical_rate     0
dtype: int64

In [19]:
len(active_flight)

5551

In [20]:
display(active_flight.sample(5))

,icao24,callsign,origin_country,longitude,latitude,velocity,geo_altitude,vertical_rate
5034,abe3e6,SWA965,United States,-112.7423,35.5644,190.76,9570.72,0.00
1911,a98386,ASA545,United States,-101.8952,30.9931,253.75,11155.68,0.00
4588,a05939,UPS799,United States,-102.2697,31.1270,280.25,8641.08,2.28
4571,0d09e5,AMX091,Mexico,134.2967,36.2542,323.79,9730.74,0.00
3691,a2c27e,SWA286,United States,-117.7710,34.0146,142.82,2819.40,-6.50


### ""SAVA DATA""

In [21]:
active_flight.to_csv('Live_Flight_data.csv' , index=False)

### Weather Data (OpenWeatherMap API)

In [ ]:
load_dotenv()

weather_api = os.getenv("WEATHER_API_KEY")

def get_weather_data(lat,lon):
  url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={weather_api}&units=metric"

  try:
      response = requests.get(url)
      if response.status_code == 200:
          data = response.json()
            
          weather_desc = data['weather'][0]['description'] 
          weather_main = data['weather'][0]['main']       
          temp = data['main']['temp']                      
          wind_speed = data['wind']['speed']              
          visibility = data.get('visibility', 10000)      
          return {
                'latitude': lat,          
                'longitude': lon,
                'weather_condition': weather_main,
                'weather_description': weather_desc,
                'temperature_c': temp,
                'wind_speed_ms': wind_speed,
                'visibility_m': visibility
            }

      else:
         return None

  except Exception as e:
     print(f"Error Fathing {e}")
     return None

In [23]:
sample_flight = active_flight.head(5).copy()

In [24]:
sample_flight.head(3)

,icao24,callsign,origin_country,longitude,latitude,velocity,geo_altitude,vertical_rate
0,80162c,AXB1672,India,90.5848,26.1838,182.38,5204.46,-7.80
1,80162d,AXB2746,India,87.0410,25.4311,267.96,8892.54,-8.45
2,801638,AXB2240,India,77.3124,28.2552,150.63,3169.92,-0.65


In [25]:
weather_list = []

print('Fetching Weather data for flight locations.....')
for index , row in sample_flight.iterrows():
  lat = row['latitude']
  lon = row['longitude']

  weather_info = get_weather_data(lat , lon)

  if weather_info:
    weather_list.append(weather_info)

  else:
    weather_list.append({
            'latitude': lat,         
            'longitude': lon,
            'weather_condition': None,
            'weather_description': None,
            'temperature_c': None,
            'wind_speed_ms': None,
            'visibility_m': None
        })
    
weather_df = pd.DataFrame(weather_list)
sample_flight = pd.concat([sample_flight.reset_index(drop=True) , weather_df] , axis=1)


Fetching Weather data for flight locations.....


In [26]:
display(sample_flight[['callsign', 'origin_country', 'latitude', 'longitude', 'weather_condition', 'weather_description', 'wind_speed_ms']])

,callsign,origin_country,latitude,latitude,longitude,longitude,weather_condition,weather_description,wind_speed_ms
0,AXB1672,India,26.1838,26.1838,90.5848,90.5848,Clear,clear sky,1.80
1,AXB2746,India,25.4311,25.4311,87.0410,87.0410,Clear,clear sky,3.13
2,AXB2240,India,28.2552,28.2552,77.3124,77.3124,Clear,clear sky,2.57
3,AXB1640,India,14.7646,14.7646,76.4250,76.4250,Clouds,overcast clouds,5.04
4,AIC7AT,India,13.5356,13.5356,77.2786,77.2786,Clouds,overcast clouds,2.85


In [27]:
sample_flight.shape

(5, 15)

### MYSQL Connection

In [62]:
load_dotenv()
con = mysql.connector.connect(
    host=os.getenv("DB_HOST", "localhost"),
    user=os.getenv("DB_USER", "root"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME", "flight_push"),
)

cursor = con.cursor()

In [63]:
load_dotenv()

con = mysql.connector.connect(
    host=os.getenv("DB_HOST", "localhost"),
    user=os.getenv("DB_USER", "root"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME", "flight_push"),
)
cursor = con.cursor()

con.commit()
print("Connection & Table creation successful!")


# FLIGHTS 

cursor.execute('''
CREATE TABLE IF NOT EXISTS flights (
    icao24 VARCHAR(50),
    callsign VARCHAR(50),
    origin_country VARCHAR(100),
    latitude DOUBLE,
    longitude DOUBLE,
    velocity DOUBLE,
    geo_altitude DOUBLE,
    vertical_rate DOUBLE,
    fetched_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

#WEATHER

cursor.execute('''
CREATE TABLE IF NOT EXISTS weather (
    latitude DOUBLE,
    longitude DOUBLE,
    weather_condition VARCHAR(100),
    weather_description VARCHAR(255),
    temperature_c DOUBLE,
    wind_speed_ms DOUBLE,
    visibility_m DOUBLE,
    fetched_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

#NEWS-ALERTS
cursor.execute('''
CREATE TABLE IF NOT EXISTS news_alerts (
    id INT AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(255),
    source VARCHAR(100),
    published_at VARCHAR(100),
    url TEXT
)
''')

con.commit()
cursor.close()
con.close()

Connection & Table creation successful!


In [ ]:
username = os.getenv("DB_USER", "root")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
database = os.getenv("DB_NAME", "flight_push")

encode_pass = urllib.parse.quote_plus(password)

engine = create_engine(f'mysql+mysqlconnector://{username}:{encode_pass}@{host}/{database}')

active_flight.to_sql('flights', con=engine, if_exists='append', index=False)

print('Done')

Done


In [31]:
weather_df.head(5)

,latitude,longitude,weather_condition,weather_description,temperature_c,wind_speed_ms,visibility_m
0,26.1838,90.5848,Clear,clear sky,28.50,1.80,10000
1,25.4311,87.0410,Clear,clear sky,27.86,3.13,10000
2,28.2552,77.3124,Clear,clear sky,23.63,2.57,10000
3,14.7646,76.4250,Clouds,overcast clouds,24.00,5.04,10000
4,13.5356,77.2786,Clouds,overcast clouds,25.11,2.85,10000


In [33]:
weather_df.shape

(5, 7)

In [32]:
weather_df.to_sql('weather', con=engine, if_exists='append', index=False)


5

### News Data (GOOGLE NEWS)

In [39]:
rss_url = "https://news.google.com/rss/search?q=aviation+weather+flight&hl=en-US&gl=US&ceid=US:en"

feed = feedparser.parse(rss_url)

news_list = []
for entry in feed.entries[:10]:
  news_list.append(
    {
        'title': entry.get('title'),
        'source': entry.get('source', {}).get('title', 'Google News'),
        'published_at': entry.get('published', str(datetime.now())),
        'url': entry.get('link')
    }
  )

news_df = pd.DataFrame(news_list)


In [40]:
news_df.head(5)

,title,source,published_at,url
0,Weather could have played a role in Greenacres...,WPEC,"Sun, 01 Nov 2026 21:38:11 GMT",https://news.google.com/rss/articles/CBMiwgFBV...
1,New aviation weather system improves US air tr...,University Corporation for Atmospheric Researc...,"Wed, 08 Apr 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMiowFBV...
2,NOAA improves aviation forecasts to bolster U....,NOAA (.gov),"Mon, 30 Mar 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMipwFBV...
3,New turbulence detection tool aims to make com...,National Weather Service (.gov),"Mon, 03 Aug 2026 18:24:36 GMT",https://news.google.com/rss/articles/CBMiUkFVX...
4,Driving decisions with aviation APIs: How real...,The Weather Company,"Thu, 29 Oct 2026 18:31:36 GMT",https://news.google.com/rss/articles/CBMixAFBV...


In [41]:
news_df.to_sql('news_alerts' , con = engine , if_exists='append' , index = False)

10

### Streamlit Deashboard

In [4]:
%%writefile app.py
import os
from dotenv import load_dotenv
import google.generativeai as genai
import pandas as pd
import plotly.express as px
import streamlit as st
from sqlalchemy import create_engine

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")

DB_URI = "sqlite:///flight_pulse.db"

st.set_page_config(page_title="Flight Pulse Dashboard", layout="wide")
st.title("✈️ Flight Pulse: Advanced Analytics Dashboard")


@st.cache_data(ttl=600)
def load_data():
  try:
    engine = create_engine(DB_URI)
    df_flights = pd.read_sql("SELECT * FROM flights", con=engine)
    df_weather = pd.read_sql("SELECT * FROM weather", con=engine)
    df_news = pd.read_sql("SELECT * FROM news_alerts", con=engine)
    return df_flights, df_weather, df_news
  except Exception as e:
    st.error(f"Database error: {e}")
    return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()


df_flights, df_weather, df_news = load_data()

if df_flights.empty:
  st.warning(
      "There was no Data!! (Please ensure flight_pulse.db is present and"
      " populated)"
  )
  filtered_df = df_flights.copy()
else:
  st.sidebar.header("🔍 Filter Options")
  countries = df_flights["origin_country"].dropna().unique().tolist()
  selected_country = st.sidebar.selectbox(
      "Select Origin Country", ["All"] + countries
  )

  if selected_country != "All":
    filtered_df = df_flights[
        df_flights["origin_country"] == selected_country
    ]
  else:
    filtered_df = df_flights.copy()

# KPIs Section
st.subheader("📊 Key Performance Indicators (KPIs)")
col1, col2, col3 = st.columns(3)

total_flights = len(filtered_df)
avg_velocity = (
    filtered_df["velocity"].mean()
    if not filtered_df.empty and "velocity" in filtered_df.columns
    else 0
)
avg_altitude = (
    filtered_df["geo_altitude"].mean()
    if not filtered_df.empty and "geo_altitude" in filtered_df.columns
    else 0
)

with col1:
  st.metric(label="Total Tracked Flights", value=total_flights)
with col2:
  st.metric(label="Average Velocity (m/s)", value=f"{avg_velocity:.2f}")
with col3:
  st.metric(label="Average Altitude (m)", value=f"{avg_altitude:.2f}")

st.markdown("---")

# Time-based Flight Activity Trend
st.subheader("📈 Time-based Flight Activity Trend")
if "fetched_at" in filtered_df.columns and not filtered_df.empty:
  trend_df = (
      filtered_df.groupby(
          pd.to_datetime(filtered_df["fetched_at"]).dt.floor("min")
      )
      .size()
      .reset_index(name="flight_count")
  )
  fig_line = px.line(
      trend_df,
      x="fetched_at",
      y="flight_count",
      markers=True,
      title="Number of Flights Tracked Over Time",
      labels={"fetched_at": "Time", "flight_count": "Active Flights"},
  )
  st.plotly_chart(fig_line, use_container_width=True)
else:
  st.info("Time series data not available for line chart.")

col_a, col_b = st.columns(2)

with col_a:
  st.subheader("📊 Flights by Origin Country")
  if not filtered_df.empty and "origin_country" in df_flights.columns:
    country_counts = (
        df_flights["origin_country"].value_counts().reset_index()
    )
    country_counts.columns = ["origin_country", "count"]

    fig_bar = px.bar(
        country_counts.head(10),
        x="origin_country",
        y="count",
        color="origin_country",
        title="Top 10 Countries by Active Flights",
        labels={"origin_country": "Country", "count": "Flight Count"},
    )
    st.plotly_chart(fig_bar, use_container_width=True)
  else:
    st.info("No data for bar chart.")

with col_b:
  st.subheader("🌦️ Weather Impact Analysis (Scatter)")
  if not df_weather.empty and "temperature_c" in df_weather.columns:
    fig_scatter = px.scatter(
        df_weather,
        x="temperature_c",
        y="wind_speed_ms",
        color="weather_condition",
        size="visibility_m" if "visibility_m" in df_weather.columns else None,
        hover_data=["weather_description"],
        title="Temperature vs Wind Speed across Weather Conditions",
        labels={
            "temperature_c": "Temperature (°C)",
            "wind_speed_ms": "Wind Speed (m/s)",
        },
    )
    st.plotly_chart(fig_scatter, use_container_width=True)
  else:
    st.info("Weather data fields not available for analysis.")

st.markdown("---")

# Live Flight Positions Map
st.subheader("🌍 Live Flight Positions Map")
if "latitude" in filtered_df.columns and "longitude" in filtered_df.columns:
  fig_map = px.scatter_geo(
      filtered_df,
      lat="latitude",
      lon="longitude",
      hover_name="callsign" if "callsign" in filtered_df.columns else None,
      color="origin_country"
      if "origin_country" in filtered_df.columns
      else None,
      projection="natural earth",
      title="Geographical Distribution of Flights",
  )
  st.plotly_chart(fig_map, use_container_width=True)
else:
  st.info("Latitude/Longitude data not available.")

st.markdown("---")
st.subheader("📰 Latest Aviation News & Alerts")

if not df_news.empty:
  for index, row in df_news.iterrows():
    with st.container():
      st.markdown(f"**🔹 [{row['title']}]({row['url']})**")
      st.caption(
          f"Source: {row['source']} | Published At: {row['published_at']}"
      )
      st.markdown("---")
else:
  st.info("No news alerts found in database.")

# Flight Pulse AI Assistant (Text-to-SQL Pipeline with Gemini)
st.markdown("---")
st.subheader("🤖 Flight Pulse AI Assistant")
st.markdown("Ask any Question (like: *'Show me the total flights'*):")

user_query = st.text_input("Ask your AI Analyst:")

if user_query:
  with st.spinner("AI is analyzing your database..."):
    try:
      genai.configure(api_key=gemini_api_key)

      schema_info = """
            Tables in database:
            - flights (icao24, callsign, origin_country, longitude, latitude, velocity, geo_altitude, vertical_rate)
            - weather (temperature_c, wind_speed_ms, weather_condition, weather_description, visibility_m, fetched_at)
            - news_alerts (title, url, source, published_at)
            """

      model = genai.GenerativeModel("gemini-2.5-pro")

      prompt = f"""
            You are an expert SQL assistant for an SQLite database.
            Based on the schema below, write ONLY a valid SQLite SQL query to answer the user's question. Do not include markdown code blocks like ```sql or any extra text, just write the raw SQL query.

            {schema_info}

            User Question: {user_query}
            """

      response = model.generate_content(prompt)
      sql_query = (
          response.text.replace("```sql", "").replace("```", "").strip()
      )

      ai_engine = create_engine(DB_URI)
      with ai_engine.connect() as connection:
        result_df = pd.read_sql(sql_query, con=connection)

      if not result_df.empty:
        summary_prompt = f"""
                The user asked: '{user_query}'
                The SQL query executed was: {sql_query}
                The query result from the database is:
                {result_df.to_string(index=False)}

                Please provide a clear, natural, and concise answer to the user based on this result.
                """
        summary_response = model.generate_content(summary_prompt)
        st.success(summary_response.text)
      else:
        st.info(
            "Query executed successfully, but no matching data found in the"
            " database."
        )

    except Exception as e:
      st.error(f"AI Error: {e}")

Overwriting app.py
